In [1]:
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime

PAGES = 5   # 5 pages = ~100 jobs

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
records = []

for page in range(1, PAGES + 1):
    url = f"https://www.naukri.com/data-analyst-jobs-{page}" if page > 1 else "https://www.naukri.com/data-analyst-jobs"
    print(f"[{page}/{PAGES}] Scraping {url}")
    driver.get(url)
    time.sleep(6)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    cards = soup.find_all("div", class_="srp-jobtuple-wrapper")
    print(f"       → {len(cards)} jobs found")

    for card in cards:
        def get(cls, tag="a"):
            el = card.find(tag, class_=cls) or card.find(class_=cls)
            return el.get_text(strip=True) if el else None

        skills_el = card.find("ul", class_="tags-gt")
        skills = ", ".join(li.get_text(strip=True) for li in skills_el.find_all("li")) if skills_el else None

        records.append({
            "Title"      : get("title"),
            "Company"    : get("comp-name"),
            "Experience" : get("expwdth", "span"),
            "Salary"     : get("sal", "span"),
            "Location"   : get("locWdth", "span"),
            "Skills"     : skills,
            "Posted"     : get("job-post-day", "span"),
            "Scraped_At" : datetime.now().strftime("%Y-%m-%d %H:%M"),
        })

driver.quit()

# ── APPEND to existing CSV instead of overwriting ────────
df = pd.DataFrame(records).drop_duplicates(subset=["Title", "Company"])

if os.path.exists("naukri_jobs.csv"):
    existing = pd.read_csv("naukri_jobs.csv")
    df = pd.concat([existing, df]).drop_duplicates(subset=["Title", "Company"]).reset_index(drop=True)
    print(f"\n📂 Existing data: {len(existing)} rows")

df.to_csv("naukri_jobs.csv", index=False)
print(f"✅ Total saved: {len(df)} jobs in naukri_jobs.csv")

[1/5] Scraping https://www.naukri.com/data-analyst-jobs
       → 20 jobs found
[2/5] Scraping https://www.naukri.com/data-analyst-jobs-2
       → 20 jobs found
[3/5] Scraping https://www.naukri.com/data-analyst-jobs-3
       → 0 jobs found
[4/5] Scraping https://www.naukri.com/data-analyst-jobs-4
       → 20 jobs found
[5/5] Scraping https://www.naukri.com/data-analyst-jobs-5
       → 0 jobs found

📂 Existing data: 178 rows
✅ Total saved: 178 jobs in naukri_jobs.csv
